# FINANCE 384 Assignment 1 – Part A

## Task A.5: Out-of-Sample Prediction Performance

This standalone notebook evaluates the predictive performance of the **pooled OLS benchmark** and the **selected Gradient Boosting model** from A.4.

Required metrics:
- pooled RMSE;
- mean monthly Spearman rank correlation;
- number of months;
- number of stock-month rows.

Gradient Boosting is evaluated on validation and test data. OLS is evaluated on the test period only. Both test models are evaluated on the same stock-month rows.


### Files required

Upload:
- `FINANCE384_assignmentA_development_panel.csv`
- `FINANCE384_stock_month_data_dictionary.csv`

The market file is not required for A.5.


In [1]:
# A.5.1 Imports and selected A.4 model settings
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor

PANEL_FILE = "FINANCE384_assignmentA_development_panel.csv"
DICTIONARY_FILE = "FINANCE384_stock_month_data_dictionary.csv"

RANDOM_STATE = 384

GB_PARAMS = {
    "loss": "squared_error",
    "n_estimators": 10,
    "learning_rate": 0.15,
    "max_depth": 1,
    "max_features": "sqrt",
    "random_state": RANDOM_STATE,
}

GB_PARAMS


{'loss': 'squared_error',
 'n_estimators': 10,
 'learning_rate': 0.15,
 'max_depth': 1,
 'max_features': 'sqrt',
 'random_state': 384}

In [2]:
# A.5.2 Load data
panel = pd.read_csv(PANEL_FILE)
data_dictionary = pd.read_csv(DICTIONARY_FILE)

panel["date"] = pd.to_datetime(panel["date"])

print("Panel shape:", panel.shape)
print("Date range:", panel["date"].min().date(), "to", panel["date"].max().date())
print("Unique stocks:", panel["permno"].nunique())
print("Duplicate stock-month rows:", panel.duplicated(["permno", "date"]).sum())


Panel shape: (198298, 27)
Date range: 1990-01-31 to 2022-12-30
Unique stocks: 1252
Duplicate stock-month rows: 0


In [3]:
# A.5.3 Define the common predictor set
numeric_predictors = [
    "size", "bm", "mom12_2", "vol12", "beta60", "ivol60",
    "turnover", "dollar_volume", "amihud_illiq", "divyield",
    "gross_profit", "roe", "asset_growth", "leverage", "accruals",
    "mkt_12m", "mkt_vol_12m", "down_market",
]

continuous_predictors = [x for x in numeric_predictors if x != "down_market"]
binary_predictors = ["down_market"]
categorical_predictors = ["ff49_code"]
feature_columns = continuous_predictors + binary_predictors + categorical_predictors

print("Numeric predictors:", len(numeric_predictors))
print("Raw feature columns:", len(feature_columns))


Numeric predictors: 18
Raw feature columns: 19


### Target construction

The target is next-month excess return \(r^e_{i,t+1}\). It is retained only when the next observation for the same stock occurs in the immediately following calendar month.


In [4]:
# A.5.4 Construct valid next-month targets
analysis = panel.sort_values(["permno", "date"]).copy()

analysis["next_date"] = analysis.groupby("permno")["date"].shift(-1)
analysis["ret_excess_t1"] = analysis.groupby("permno")["ret_excess_t"].shift(-1)

analysis["is_consecutive_next_month"] = (
    analysis["next_date"].dt.to_period("M")
    == analysis["date"].dt.to_period("M") + 1
)

analysis.loc[~analysis["is_consecutive_next_month"], "ret_excess_t1"] = np.nan
analysis_valid = analysis.loc[analysis["ret_excess_t1"].notna()].copy()

print("Rows with valid next-month targets:", len(analysis_valid))


Rows with valid next-month targets: 197016


In [5]:
# A.5.5 Apply fixed chronological splits
train = analysis_valid.loc[
    (analysis_valid["date"] >= "1990-01-01")
    & (analysis_valid["date"] <= "2014-12-31")
].copy()

validation = analysis_valid.loc[
    (analysis_valid["date"] >= "2015-01-01")
    & (analysis_valid["date"] <= "2018-12-31")
].copy()

test = analysis_valid.loc[
    (analysis_valid["date"] >= "2019-01-01")
    & (analysis_valid["date"] <= "2022-11-30")
].copy()

ols_estimation = pd.concat([train, validation], axis=0).sort_values(
    ["date", "permno"]
).reset_index(drop=True)

pd.DataFrame({
    "Sample": ["Training", "Validation", "OLS estimation", "Test"],
    "Months": [
        train["date"].dt.to_period("M").nunique(),
        validation["date"].dt.to_period("M").nunique(),
        ols_estimation["date"].dt.to_period("M").nunique(),
        test["date"].dt.to_period("M").nunique(),
    ],
    "Rows": [len(train), len(validation), len(ols_estimation), len(test)],
})


,Sample,Months,Rows
0,Training,300,149334
1,Validation,48,24051
2,OLS estimation,348,173385
3,Test,47,23631


### Common preprocessing

Preprocessing is fitted on the training sample only and then applied unchanged to validation, OLS-estimation, and test data.


In [6]:
# A.5.6 Fit common preprocessing on training data
continuous_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

binary_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
])

categorical_pipeline = Pipeline(steps=[
    ("onehot", OneHotEncoder(
        drop="first",
        handle_unknown="ignore",
        sparse_output=False
    )),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("continuous", continuous_pipeline, continuous_predictors),
        ("binary", binary_pipeline, binary_predictors),
        ("industry", categorical_pipeline, categorical_predictors),
    ],
    remainder="drop",
)

X_train_raw = train[feature_columns]
X_validation_raw = validation[feature_columns]
X_test_raw = test[feature_columns]
X_ols_estimation_raw = ols_estimation[feature_columns]

y_train = train["ret_excess_t1"].to_numpy()
y_validation = validation["ret_excess_t1"].to_numpy()
y_test = test["ret_excess_t1"].to_numpy()
y_ols_estimation = ols_estimation["ret_excess_t1"].to_numpy()

preprocessor.fit(X_train_raw)

X_train = preprocessor.transform(X_train_raw)
X_validation = preprocessor.transform(X_validation_raw)
X_test = preprocessor.transform(X_test_raw)
X_ols_estimation = preprocessor.transform(X_ols_estimation_raw)

print("Training matrix:", X_train.shape)
print("Validation matrix:", X_validation.shape)
print("Test matrix:", X_test.shape)
print("OLS estimation matrix:", X_ols_estimation.shape)


Training matrix: (149334, 64)
Validation matrix: (24051, 64)
Test matrix: (23631, 64)
OLS estimation matrix: (173385, 64)


## Reconstruct the assessed models

- Pooled OLS: fit on training + validation.
- Gradient Boosting: fit on training only using the A.4-selected specification.


In [7]:
# A.5.7 Fit pooled OLS and generate test predictions
ols_model = LinearRegression(fit_intercept=True)
ols_model.fit(X_ols_estimation, y_ols_estimation)

ols_test_pred = ols_model.predict(X_test)

print("OLS test predictions:", len(ols_test_pred))
print("Missing OLS test predictions:", int(np.isnan(ols_test_pred).sum()))


OLS test predictions: 23631
Missing OLS test predictions: 0


In [8]:
# A.5.8 Fit selected Gradient Boosting model on training only
gb_model = GradientBoostingRegressor(**GB_PARAMS)
gb_model.fit(X_train, y_train)

gb_validation_pred = gb_model.predict(X_validation)
gb_test_pred = gb_model.predict(X_test)

print("GB validation predictions:", len(gb_validation_pred))
print("GB test predictions:", len(gb_test_pred))
print("Missing GB validation predictions:", int(np.isnan(gb_validation_pred).sum()))
print("Missing GB test predictions:", int(np.isnan(gb_test_pred).sum()))


GB validation predictions: 24051
GB test predictions: 23631
Missing GB validation predictions: 0
Missing GB test predictions: 0


## Required metrics

Pooled RMSE is calculated across all valid stock-month rows in the relevant period.

Mean monthly Spearman correlation is calculated by ranking predicted and realised returns within each month using **average ranks for ties**, computing the correlation between those ranks, and then averaging the monthly correlations.


In [9]:
# A.5.9 Evaluation functions
def pooled_rmse(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    return float(np.sqrt(np.mean((actual - predicted) ** 2)))


def monthly_spearman_table(df, actual_col, predicted_col):
    rows = []
    for date, month_df in df.groupby("date", sort=True):
        temp = month_df[[actual_col, predicted_col]].dropna().copy()

        actual_rank = temp[actual_col].rank(method="average")
        predicted_rank = temp[predicted_col].rank(method="average")

        spearman = actual_rank.corr(predicted_rank)

        rows.append({
            "date": date,
            "n_stocks": len(temp),
            "spearman": spearman,
        })

    return pd.DataFrame(rows)


def evaluate_predictions(df, actual_col, predicted_col):
    valid = df[[actual_col, predicted_col]].dropna()

    monthly_spearman = monthly_spearman_table(
        df,
        actual_col,
        predicted_col
    )

    return {
        "rmse": pooled_rmse(valid[actual_col], valid[predicted_col]),
        "mean_monthly_spearman": float(monthly_spearman["spearman"].mean()),
        "months": int(monthly_spearman["date"].nunique()),
        "stock_month_rows": int(len(valid)),
        "monthly_spearman_table": monthly_spearman,
    }


In [10]:
# A.5.10 Assemble validation and common test prediction datasets
gb_validation_predictions = validation[
    ["date", "permno", "ticker", "ret_excess_t1"]
].copy()

gb_validation_predictions = gb_validation_predictions.rename(
    columns={"ret_excess_t1": "actual_excess_return_t1"}
)
gb_validation_predictions["gb_pred_excess_return_t1"] = gb_validation_pred

test_predictions = test[
    ["date", "permno", "ticker", "ret_excess_t1"]
].copy()

test_predictions = test_predictions.rename(
    columns={"ret_excess_t1": "actual_excess_return_t1"}
)
test_predictions["ols_pred_excess_return_t1"] = ols_test_pred
test_predictions["gb_pred_excess_return_t1"] = gb_test_pred

print("Common test rows:", len(test_predictions))
print("Missing OLS predictions:",
      int(test_predictions["ols_pred_excess_return_t1"].isna().sum()))
print("Missing GB predictions:",
      int(test_predictions["gb_pred_excess_return_t1"].isna().sum()))


Common test rows: 23631
Missing OLS predictions: 0
Missing GB predictions: 0


In [11]:
# A.5.11 Calculate required validation/test metrics
gb_validation_metrics = evaluate_predictions(
    gb_validation_predictions,
    "actual_excess_return_t1",
    "gb_pred_excess_return_t1",
)

gb_test_metrics = evaluate_predictions(
    test_predictions,
    "actual_excess_return_t1",
    "gb_pred_excess_return_t1",
)

ols_test_metrics = evaluate_predictions(
    test_predictions,
    "actual_excess_return_t1",
    "ols_pred_excess_return_t1",
)

a5_results = pd.DataFrame([
    {
        "Model": "Gradient Boosting",
        "Period": "Validation",
        "RMSE": gb_validation_metrics["rmse"],
        "Mean monthly Spearman": gb_validation_metrics["mean_monthly_spearman"],
        "Months": gb_validation_metrics["months"],
        "Stock-month rows": gb_validation_metrics["stock_month_rows"],
    },
    {
        "Model": "Gradient Boosting",
        "Period": "Test",
        "RMSE": gb_test_metrics["rmse"],
        "Mean monthly Spearman": gb_test_metrics["mean_monthly_spearman"],
        "Months": gb_test_metrics["months"],
        "Stock-month rows": gb_test_metrics["stock_month_rows"],
    },
    {
        "Model": "Pooled OLS",
        "Period": "Test",
        "RMSE": ols_test_metrics["rmse"],
        "Mean monthly Spearman": ols_test_metrics["mean_monthly_spearman"],
        "Months": ols_test_metrics["months"],
        "Stock-month rows": ols_test_metrics["stock_month_rows"],
    },
])

a5_results


,Model,Period,RMSE,Mean monthly Spearman,Months,Stock-month rows
0,Gradient Boosting,Validation,0.076816,0.004537,48,24051
1,Gradient Boosting,Test,0.100331,-0.006536,47,23631
2,Pooled OLS,Test,0.100299,0.017157,47,23631


### Direct test comparison

Lower RMSE is preferred. Higher mean monthly Spearman indicates stronger cross-sectional ranking ability.


In [12]:
# A.5.12 Direct model comparison
test_comparison = pd.DataFrame({
    "Comparison": [
        "GB test RMSE minus OLS test RMSE",
        "GB RMSE improvement vs OLS (%)",
        "GB test Spearman minus OLS test Spearman",
        "GB validation RMSE minus GB test RMSE",
        "GB validation Spearman minus GB test Spearman",
    ],
    "Value": [
        gb_test_metrics["rmse"] - ols_test_metrics["rmse"],
        100 * (
            ols_test_metrics["rmse"] - gb_test_metrics["rmse"]
        ) / ols_test_metrics["rmse"],
        gb_test_metrics["mean_monthly_spearman"]
        - ols_test_metrics["mean_monthly_spearman"],
        gb_validation_metrics["rmse"] - gb_test_metrics["rmse"],
        gb_validation_metrics["mean_monthly_spearman"]
        - gb_test_metrics["mean_monthly_spearman"],
    ],
})

test_comparison


,Comparison,Value
0,GB test RMSE minus OLS test RMSE,0.000033
1,GB RMSE improvement vs OLS (%),-0.032535
2,GB test Spearman minus OLS test Spearman,-0.023693
3,GB validation RMSE minus GB test RMSE,-0.023515
4,GB validation Spearman minus GB test Spearman,0.011073


In [13]:
# A.5.13 Retain monthly Spearman series for audit
gb_validation_monthly_spearman = gb_validation_metrics["monthly_spearman_table"]

gb_test_monthly_spearman = gb_test_metrics["monthly_spearman_table"].rename(
    columns={"spearman": "gb_spearman"}
)

ols_test_monthly_spearman = ols_test_metrics["monthly_spearman_table"].rename(
    columns={"spearman": "ols_spearman"}
)

test_monthly_rank_comparison = gb_test_monthly_spearman.merge(
    ols_test_monthly_spearman[["date", "ols_spearman"]],
    on="date",
    how="inner",
)

test_monthly_rank_comparison.head()


,date,n_stocks,gb_spearman,ols_spearman
0,2019-01-31,502,-0.038719,0.024831
1,2019-02-28,503,0.074580,-0.161517
2,2019-03-29,502,-0.160567,0.038086
3,2019-04-30,504,-0.036934,-0.152108
4,2019-05-31,503,0.050039,0.201445


## A.5 Summary

The selected Gradient Boosting model is evaluated on validation and test periods, while pooled OLS is evaluated on the test period. Both test models use identical stock-month observations.

The notebook reports pooled RMSE, mean monthly Spearman rank correlation, month counts, and stock-month row counts. These prediction results should be interpreted jointly with the portfolio evidence developed in A.6.
